In [ ]:
import base64
import gzip
import json
import pickle
import pyrender
import numpy as np
import random
import secrets
import trimesh
import voxeloo
import zlib

from PIL import Image
from dataclasses import dataclass
from matplotlib import pyplot as plt
from scipy.ndimage import (
    distance_transform_edt as edt,
    gaussian_filter as blur, 
    maximum_filter as max_filter,
)
from tqdm.notebook import tqdm
from typing import Any, List, Optional, Dict, Tuple
from scipy import ndimage

In [ ]:
CHUNK_PADDING = 64
CHUNK_DIM = (2048, 2048)
CHUNK_ORIGIN = (-2048, -2048)

In [ ]:
MAP_DIM = (CHUNK_DIM[0] + 2 * CHUNK_PADDING, CHUNK_DIM[1] + 2 * CHUNK_PADDING)
MAP_ORIGIN = (CHUNK_ORIGIN[0] - CHUNK_PADDING, CHUNK_ORIGIN[1] - CHUNK_PADDING)

### Visualization routines

In [ ]:
def heatmap_to_image(heat):
    return Image.fromarray(np.uint8(heat * 255) , "L")

In [ ]:
def visualize_voxels(voxels, wireframe=False):
    if voxels.dtype == bool:
        voxels = np.array([0, 0xffffffff])[voxels.astype(int)]
    
    mesh = voxeloo.voxels.voxels_to_mesh(voxels)
    
    # Convert the mesh into the trimesh format.
    tm = trimesh.Trimesh(
        vertices=mesh.vertices[:, 0:3],
        faces=mesh.triangles,
    )
    
    # Add the vertex colors.
    tm.visual.face_colors = mesh.vertices[mesh.triangles[:, 0], 3:6]
    
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2, 1.0])
    scene.add(
        pyrender.Mesh.from_trimesh(
            tm,
            smooth=False,
            wireframe=wireframe,
        )
    )
    pyrender.Viewer(
        scene,
        use_raymond_lighting=True,
    )

### Routines

In [ ]:
def masked_blur(x, mask, radius):
    mask = mask.astype(float)
    return blur(mask * x, radius) / (blur(mask, radius) + 0.0001)

### Load surface

In [ ]:
@dataclass
class TerrainVoxel:
    name: str
    index: int
    color: int

In [ ]:
def to_flora_id(terrain_id):
    return (1 << 24) | (terrain_id & 0xffffff)

# These IDs match the terrain IDs in: src/galois/js/assets/blocks.ts
terrain = [
    # Blocks
    TerrainVoxel("none", 0, 0x00000000),
    TerrainVoxel("basalt", 8, 0x332c2bff),
    TerrainVoxel("bedrock", 6, 0x4e4a54ff),
    TerrainVoxel("birch_log", 12, 0xd4cdc5ff),
    TerrainVoxel("clay", 17, 0x6e554fff),
    TerrainVoxel("coal_ore", 19, 0x282e2aff),
    TerrainVoxel("cobblestone", 5, 0x524535ff),
    TerrainVoxel("diamond_ore", 23, 0x89c5d9ff),
    TerrainVoxel("dirt", 2, 0x543919ff),
    TerrainVoxel("gold_ore", 11, 0xdecb52ff),
    TerrainVoxel("granite", 13, 0x636d70ff),
    TerrainVoxel("grass", 1, 0x319436ff),
    TerrainVoxel("gravel", 36, 0xab9865ff),
    TerrainVoxel("hay", 35, 0xa39f33ff),
    TerrainVoxel("limestone", 9, 0xc3c4b1ff),
    TerrainVoxel("moss", 40, 0x30784eff),
    TerrainVoxel("neptunium_ore", 30, 0x1d4030ff),
    TerrainVoxel("oak_log", 3, 0x4d442cff),
    TerrainVoxel("pumpkin", 10, 0xc78e08ff),
    TerrainVoxel("quartzite", 7, 0xa89c8aff),
    TerrainVoxel("rubber_log", 14, 0x94570cff),
    TerrainVoxel("silver_ore", 25, 0xd8e3e3ff),
    TerrainVoxel("stone", 4, 0xc1c9c8ff),
    TerrainVoxel("snow", 37, 0xf5f5f5ff),
    
    # Flora
    TerrainVoxel("oak_leaf", to_flora_id(1), 0x234a2dff),
    TerrainVoxel("birch_leaf", to_flora_id(2), 0x586b23ff),
    TerrainVoxel("rubber_leaf", to_flora_id(3), 0x176e47ff),
    TerrainVoxel("switch_grass", to_flora_id(4), 0x1ad94aff),
    TerrainVoxel("azalea_flower", to_flora_id(5), 0xb84f9dff),
    TerrainVoxel("bell_flower", to_flora_id(6), 0x5474a8ff),
    TerrainVoxel("dandelion_flower", to_flora_id(7), 0xadb020ff),
    TerrainVoxel("daylily_flower", to_flora_id(8), 0xbf5a17ff),
    TerrainVoxel("lilac_flower", to_flora_id(9), 0x742fb5ff),
    TerrainVoxel("rose_flower", to_flora_id(10), 0xb51200ff),
    TerrainVoxel("cotton_flower", to_flora_id(11), 0xd1c9c7ff),
    TerrainVoxel("hemp_bush", to_flora_id(12), 0x18381eff),
    TerrainVoxel("red_mushroom", to_flora_id(13), 0x82283fff),
]

terrain_index = {tv.name: tv.index for tv in terrain}

terrain_color = np.zeros(max(tv.index for tv in terrain) + 1, dtype=np.uint32)
for tv in terrain:
    terrain_color[tv.index] = tv.color

In [ ]:
@dataclass
class RegionMap:
    ids: Dict[str, int]
    assignments: np.ndarray
    colors: List[int]
    radii: List[int]
        
@dataclass
class HeightMap:
    data: np.ndarray
        
@dataclass
class Column:
    name: str
    samples: List[np.ndarray]
    color: int

@dataclass
class SurfaceMap:
    ids: Dict[str, int]
    assignments: np.ndarray
    columns: List[Column]
        
@dataclass
class ColumnMap:
    data: np.ndarray

In [ ]:
@dataclass
class Surface:
    rm: Any
    hm: Any
    sm: Any
    cm: Any

In [ ]:
ROOT_DIR = "C:/Users/taylor/data/biomes/"

def load_surface(chunk):
    name = "_".join(map(str, chunk))
    with gzip.open(f"{ROOT_DIR}/alpha_world/surface/regions_{name}.dump", "rb") as f:
        rm = pickle.load(f)
    with gzip.open(f"{ROOT_DIR}/alpha_world/surface/heights_{name}.dump", "rb") as f:
        hm = pickle.load(f)
    with gzip.open(f"{ROOT_DIR}/alpha_world/surface/surface_{name}.dump", "rb") as f:
        sm = pickle.load(f)
    with gzip.open(f"{ROOT_DIR}/alpha_world/surface/columns_{name}.dump", "rb") as f:
        cm = pickle.load(f)
    return Surface(rm, hm, sm, cm)

In [ ]:
surface = load_surface(np.array(CHUNK_ORIGIN) // np.array(CHUNK_DIM))

In [ ]:
Image.fromarray(np.array(surface.rm.colors, dtype=np.uint8)[surface.rm.assignments])

In [ ]:
canyons = surface.rm.assignments == surface.rm.ids["canyons"]
canyons = np.logical_or(canyons, blur(canyons.astype(float), 8) > 0.2)
heatmap_to_image(canyons)

In [ ]:
heights = np.zeros(surface.hm.data.shape)
heights[canyons] = surface.hm.data[canyons] + 4
heights[canyons] = masked_blur(heights, canyons, 64.0)[canyons]

In [ ]:
print(np.mean(heights[canyons] - surface.hm.data[canyons]))
print(np.max(heights[canyons] - surface.hm.data[canyons]))
print(np.min(heights[canyons] - surface.hm.data[canyons]))

In [ ]:
%%time

hm = surface.hm.data.astype(int) + 256
wm = heights.astype(int) + 256

voxels = np.zeros(shape=(CHUNK_DIM[0], 512, CHUNK_DIM[1]), dtype=np.uint32)

for z in range(voxels.shape[0]):
    for x in range(voxels.shape[2]):
        h = hm[z, x]
        voxels[z, :h, x] = 0xffffffff
        if canyons[z, x]:
            w = wm[z, x]
            voxels[z, h:w, x] = 0x0000ffff

In [ ]:
#visualize_voxels(voxels)

In [ ]:
%%time 

water_tensor = voxeloo.tensors.Tensor_U8(
    shape=(CHUNK_DIM[0], 512, CHUNK_DIM[1]),
    fill=0,
)

In [ ]:
%%time

z, y, x = np.where(voxels == 0x0000ffff)

In [ ]:
%%time
    
# Assign to the compressed tensor.
pos = np.stack([x, y, z], axis=-1)
val = 15 * np.ones(x.shape[0], dtype=np.uint8)
water_tensor.assign(pos, val)

In [ ]:
water_tensor.get(x[0], y[0], z[0])

In [ ]:
%%time

recovered_voxels = water_tensor.array().astype(np.uint32)
recovered_voxels[recovered_voxels == 15] = 0x0000ffff

In [ ]:
#visualize_voxels(recovered_voxels)

In [ ]:
chunk_name = "_".join([str(d) for d in np.array(CHUNK_ORIGIN) // np.array(CHUNK_DIM)])

with open(f"C:/Users/taylor/code/biomes/src/galois/data/gaia/water_{chunk_name}.tensor", "wb") as f:
    blob = water_tensor.dump()
    f.write(blob)